Chain of thought prompting

In [1]:
!pip install transformers torch

In [2]:
from transformers import pipeline

In [3]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

In [4]:
def chain_of_thought_prompt(user_question):
    prompt = f"""
Solve the following problem step by step and explain your reasoning clearly.

Question:
{user_question}

Answer:
"""
    return prompt

In [5]:
def run_cot(question):
    prompt = chain_of_thought_prompt(question)

    response = generator(
        prompt,
        max_length=200,
        do_sample=False
    )

    print("USER QUESTION:")
    print(question)
    print("\nCHAIN OF THOUGHT RESPONSE:\n")
    print(response[0]["generated_text"])

In [6]:
question = "A student studies 4 hours per day for 6 days. How many hours does the student study in total?"

run_cot(question)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER QUESTION:
A student studies 4 hours per day for 6 days. How many hours does the student study in total?

CHAIN OF THOUGHT RESPONSE:


Solve the following problem step by step and explain your reasoning clearly.

Question:
A student studies 4 hours per day for 6 days. How many hours does the student study in total?

Answer:
 in total so he studies 96 + 96 = 108 hours. The answer: 108.


Tree of thought prompting

In [7]:
!pip install transformers torch

In [8]:
#load model from hugging face
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def generate_text(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_text("Solve step by step: What is 12 * 8?"))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


3 years are 5–5 hours longer according theres=1 night (Wk 15); however. Thus yearly delay 4+ nights refers total period spent


In [9]:
#helper function: generate text
def generate_text(prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [10]:
#Step 1: Generate multiple thoughts (branches)
def generate_thoughts(question, k=3):
    thoughts = []
    for i in range(k):
        prompt = f"""
Question: {question}
Think step by step and suggest one possible reasoning path.
Thought:
"""
        thought = generate_text(prompt)
        thoughts.append(thought)
    return thoughts

In [11]:
# Step 2: Expand each thought (Tree growth 🌳)
def expand_thought(thought):
    prompt = f"""
Here is a reasoning step:
{thought}

Continue this reasoning logically:
"""
    return generate_text(prompt)

In [12]:
#Step 3: Score thoughts (simple heuristic)
#we score by length + presence of numbers / logic words.
def score_thought(thought):
    score = len(thought)
    logic_keywords = ["because", "therefore", "so", "hence", "thus"]

    for word in logic_keywords:
        if word in thought.lower():
            score += 20
    return score

In [13]:
#Step 4: Tree of Thought solver
def tree_of_thought_solver(question, branches=3):
    print("🌱 Generating initial thoughts...\n")
    thoughts = generate_thoughts(question, branches)

    expanded = []
    for t in thoughts:
        e = expand_thought(t)
        expanded.append(e)

    scored = [(score_thought(t), t) for t in expanded]
    scored.sort(reverse=True, key=lambda x: x[0])

    best_thought = scored[0][1]

    final_prompt = f"""
Based on the reasoning below, give the final answer clearly.

Reasoning:
{best_thought}

Final Answer:
"""

    final_answer = generate_text(final_prompt)
    return final_answer

In [14]:
question = "If a train travels 60 km in 1 hour and then 30 km in 30 minutes, what is its average speed?"

answer = tree_of_thought_solver(question)
print("✅ Final Answer:\n", answer)

🌱 Generating initial thoughts...

✅ Final Answer:
 a 3x p = 3 while 20% increased to be 25,5=2+20 = 1225 so 2520 / 2425 equal 1285 cm is = 1 to 23 + (32+249) in terms percentage and 225 has 20 + 300 total is 1560x 3080 and 3585mm in ratio = 716*568 is 7.5 and 380
